# Transformer (DistilBERT) 감정분석 — GLUE/SST-2

In [1]:
# torchvision과 torchaudio를 추가하여 torch 버전과 호환되도록 함께 업데이트합니다.
# !pip -q install -U "torch>=2.2,<3.0" "torchvision" "torchaudio" "datasets>=3.0.1" "transformers>=4.45.2" "accelerate>=1.0.1" "evaluate>=0.4.2"

import torch, transformers, datasets, evaluate
import numpy as np

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("Transformers:", transformers.__version__, "| Datasets:", datasets.__version__)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

/home/kim/miniconda3/envs/ai_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.10.0+cu130 | CUDA: True
Transformers: 5.2.0 | Datasets: 4.5.0
Using device: cuda


In [2]:
from datasets import load_dataset
ds = load_dataset("glue", "sst2")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
import torch

# 1. 환경 설정
MODEL = "distilbert-base-uncased"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL)

# 3. 전처리 함수 (return_tensors는 여기서 하지 않고 collator에게 맡깁니다)
def preprocess(ex):
    return tokenizer(ex["sentence"], truncation=True, max_length=256)

# 4. 데이터셋 매핑 (ds가 정의되어 있다고 가정)
# remove_columns에 "label"은 포함하지 않도록 주의하세요!
enc = ds.map(preprocess, batched=True, remove_columns=["sentence", "idx"])

# 5. 데이터 콜레이터 (다이내믹 패딩 적용)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 6. 모델 로드 (이때 발생하는 Warning은 무시해도 됩니다)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).to(device)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2408.68it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
import evaluate
from transformers import TrainingArguments, Trainer
import torch

# 지표 로드
acc = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def metrics(p):
    predictions, labels = p
    # predictions가 튜플로 나오는 경우(logits 외 다른 요소가 있는 경우)를 대비해 분기 처리하거나
    # 일반적인 경우 predictions 자체가 logits입니다.
    # 여기서는 numpy 배열이라고 가정하고 처리합니다.
    preds = predictions.argmax(-1)

    return {
        "acc": acc.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"]
    }

# TrainingArguments 설정
args = TrainingArguments(
    output_dir="./sst2_2025",
    eval_strategy="epoch",          # 수정됨: evaluation_strategy -> eval_strategy
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    learning_rate=2e-5,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(), # GPU가 있을 때만 fp16 사용
    report_to="none",
    seed=2025
)

# fp16: 16bit / (default) 32bit float >> 16bit로 낮춤
# A100 이상 돌리고 싶다면 bf16=True 사용 권고 (bf: brain floating point)

# Trainer 초기화
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=enc["train"],
    eval_dataset=enc["validation"],
    data_collator=data_collator,
    compute_metrics=metrics
)

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,Acc,F1
1,0.191521,0.295559,0.910550,0.913525
2,0.121063,0.356289,0.908257,0.911504


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.96it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=8420, training_loss=0.17636005600954177, metrics={'train_runtime': 160.69, 'train_samples_per_second': 838.248, 'train_steps_per_second': 52.399, 'total_flos': 1224860836574256.0, 'train_loss': 0.17636005600954177, 'epoch': 2.0})

In [18]:
txt=["This movie was amazing!","Worst film ever."]
inp=tokenizer(txt,return_tensors="pt",padding=True,truncation=True,max_length=256).to(model.device)
with torch.no_grad(): out=torch.softmax(model(**inp).logits,dim=-1).cpu().numpy()
for t,p in zip(txt,out): print(f"{t}\n→ Negative={p[0]:.3f}, Positive={p[1]:.3f}")

This movie was amazing!
→ Negative=0.001, Positive=0.999
Worst film ever.
→ Negative=0.997, Positive=0.003
